In [9]:
# Import required libraries
import os
import pandas as pd

In [10]:
# Load the dataset
df = pd.read_csv('../data/train_flow.tsv', sep='\t', low_memory=False)
df = df.drop(columns=['flow_id'])
df.head()

,participant_id,timepoint,parent_population,population_defnition,name,unit,value,material
0,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+,Freq. of,WBC",live CD45+ cells?,cells/uL,176.220000,blood
1,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ1: CD27,, IgD+,Freq. of,C...",Naive B cells,cells/uL,112.780800,blood
2,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ2: CD27+,, IgD+,Freq. of,...",Memory class switched + non-class switched B c...,cells/uL,12.370644,blood
3,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ3: CD27+,, IgD,Freq. of,C...",Memory class switched B cells,cells/uL,21.322620,blood
4,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ4: CD27,, IgD,Freq. of,CD19+",B cells,cells/uL,29.781180,blood


In [11]:
# Rename typo column
df = df.rename(columns={'population_defnition': 'population_definition'})

# Clean material column (standardize labels)
df['material'] = df['material'].replace({
    'specimen type: other': 'Other'
})

# Clean name column (remove trailing '?', strip whitespace)
# Note: do NOT use .str.capitalize() — it corrupts acronyms like "CD4" → "Cd4"
df['name'] = df['name'].str.rstrip('?').str.strip()

df.head()

,participant_id,timepoint,parent_population,population_definition,name,unit,value,material
0,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+,Freq. of,WBC",live CD45+ cells,cells/uL,176.220000,blood
1,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ1: CD27,, IgD+,Freq. of,C...",Naive B cells,cells/uL,112.780800,blood
2,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ2: CD27+,, IgD+,Freq. of,...",Memory class switched + non-class switched B c...,cells/uL,12.370644,blood
3,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ3: CD27+,, IgD,Freq. of,C...",Memory class switched B cells,cells/uL,21.322620,blood
4,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ4: CD27,, IgD,Freq. of,CD19+",B cells,cells/uL,29.781180,blood


In [12]:
# filter to these 4 columns, rest are noise
df = df[['participant_id', 'timepoint', 'name', 'value']]

In [13]:
# Print all columns and their unique values
for col in df.columns:
    print(f"{col}: {df[col].unique()}\n")

participant_id: <ArrowStringArray>
['SDY180.SUB119292', 'SDY180.SUB119293', 'SDY180.SUB119294',
 'SDY180.SUB119295', 'SDY180.SUB119296', 'SDY180.SUB119297',
 'SDY180.SUB119262', 'SDY180.SUB119264', 'SDY180.SUB119266',
 'SDY180.SUB119267',
 ...
  'SDY80.SUB114500',  'SDY80.SUB114502',  'SDY80.SUB114503',
  'SDY80.SUB114504',  'SDY80.SUB114506',  'SDY80.SUB114507',
  'SDY80.SUB114508',  'SDY80.SUB114510',  'SDY80.SUB114511',
  'SDY80.SUB114512']
Length: 264, dtype: str

timepoint: [ -7.    0.    1.   10.   14.   21.   28.    3.    7.    0.5   2.    4.
   5.    6.    8.    9.  120.  240.   -5.   90.   70. ]

name: <ArrowStringArray>
[                                  'live CD45+ cells',
                                      'Naive B cells',
 'Memory class switched + non-class switched B cells',
                      'Memory class switched B cells',
                                            'B cells',
                                                  nan,
                             'Ab

In [14]:
agg = df.groupby(['participant_id', 'timepoint', 'name'], as_index=False)['value'].mean()
agg = agg[agg['timepoint'].isin([0, 7])]

pivot = agg.pivot_table(index='participant_id', columns=['timepoint', 'name'], values='value')

# Match transcriptomics pattern: d{timepoint}_{name}
pivot.columns = [f'd{int(t)}_{n}' for t, n in pivot.columns]
pivot = pivot.reset_index()

In [ ]:
pivot.head()

,participant_id,d0_Ab-secreting cells ASC,d0_B cells,d0_CD4 T cells,d0_CD4 Tcn,d0_CD4 Tem,d0_CD4 Tem + CD4 Temra,d0_CD56hi + CD56+ NK cells,d0_CD8 T cells,d0_CD8 Tcm,...,d7_Naive B cells,d7_Naive CD4 + Naive CD8 T cells,d7_Neutrophils,d7_Neutrophils + Basophils,d7_Plasmacytoid DCs,d7_T cells,d7_Tfh CXCR5+ CD4 T cell,d7_Th1 CXCR3+ CD4 T cells,d7_live CD45+ cells,d7_neutrophils
0,SDY180.SUB119262,0.251182,38.158720,197.608956,85.42314,120.80040,11.936230,NaN,76.750338,1.281588,...,145.66635,74.436600,2520.0,NaN,NaN,984.20,30.197109,40.811781,74.331370,NaN
1,SDY180.SUB119264,1.059942,55.077360,399.718881,177.78852,139.36536,13.936536,NaN,145.225872,1.625184,...,121.22550,240.831570,3943.2,NaN,NaN,1033.41,28.276639,26.715177,109.635096,NaN
2,SDY180.SUB119266,1.272667,42.161190,683.230790,223.05360,307.24540,41.221180,NaN,308.526570,3.989700,...,44.85912,468.859475,2154.0,NaN,NaN,2101.52,58.363941,89.179673,113.073236,NaN
3,SDY180.SUB119267,0.372593,55.266000,528.745645,110.10560,171.17980,59.525840,NaN,153.215050,2.616980,...,198.71640,376.872880,2371.5,NaN,NaN,1635.64,24.030528,32.635522,83.993819,NaN
4,SDY180.SUB119268,0.027097,13.336092,540.570923,143.82182,276.00488,27.184818,NaN,318.191692,1.817343,...,153.59976,211.104810,NaN,NaN,NaN,1504.89,20.550982,32.203297,20.246370,NaN


In [18]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
pivot.to_csv('../cleaned_data/flow_cleaned.csv', index=False)